# 01 — Data Ingestion

## Brazilian E-Commerce Analytics Portfolio

This notebook loads and validates the raw Olist e-commerce datasets used throughout the project.

### Objectives

1. Locate all required raw CSV files.
2. Load the datasets without modifying the source files.
3. Validate expected columns and primary identifiers.
4. Parse date and timestamp fields.
5. Create a dataset inventory and schema-validation report.
6. Save validation artifacts in `data/processed/`.

> All project files, comments, outputs, and documentation are written in English.

## Expected raw files

Download the **Brazilian E-Commerce Public Dataset by Olist** and place these files in `data/raw/`:

- `olist_customers_dataset.csv`
- `olist_orders_dataset.csv`
- `olist_order_items_dataset.csv`
- `olist_order_payments_dataset.csv`
- `olist_order_reviews_dataset.csv`
- `olist_products_dataset.csv`
- `olist_sellers_dataset.csv`
- `product_category_name_translation.csv`

The geolocation file is optional for the first version:

- `olist_geolocation_dataset.csv`

The original files must remain unchanged. Generated validation artifacts are written to `data/processed/`.

In [ ]:
from pathlib import Path
from typing import Dict, Iterable

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)

print(f"Pandas version: {pd.__version__}")

## 1. Resolve project directories

The next cell allows the notebook to run from either the repository root or the `notebooks/` directory.

In [ ]:
def find_project_root(start_path: Path) -> Path:
    """Find the nearest parent directory containing data/raw."""
    start_path = start_path.resolve()

    for candidate in [start_path, *start_path.parents]:
        if (candidate / "data" / "raw").exists():
            return candidate

    raise FileNotFoundError(
        "Project root not found. Create data/raw and run the notebook "
        "from the repository root or notebooks directory."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root:       {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Output directory:   {PROCESSED_DATA_DIR}")

## 2. Define expected files and minimum schemas

Additional columns are allowed, but all required columns must be present.

In [ ]:
EXPECTED_SCHEMAS: Dict[str, set[str]] = {
    "customers": {
        "customer_id", "customer_unique_id", "customer_zip_code_prefix",
        "customer_city", "customer_state"
    },
    "orders": {
        "order_id", "customer_id", "order_status", "order_purchase_timestamp",
        "order_approved_at", "order_delivered_carrier_date",
        "order_delivered_customer_date", "order_estimated_delivery_date"
    },
    "order_items": {
        "order_id", "order_item_id", "product_id", "seller_id",
        "shipping_limit_date", "price", "freight_value"
    },
    "payments": {
        "order_id", "payment_sequential", "payment_type",
        "payment_installments", "payment_value"
    },
    "reviews": {
        "review_id", "order_id", "review_score", "review_comment_title",
        "review_comment_message", "review_creation_date",
        "review_answer_timestamp"
    },
    "products": {
        "product_id", "product_category_name", "product_name_lenght",
        "product_description_lenght", "product_photos_qty",
        "product_weight_g", "product_length_cm",
        "product_height_cm", "product_width_cm"
    },
    "sellers": {
        "seller_id", "seller_zip_code_prefix", "seller_city", "seller_state"
    },
    "category_translation": {
        "product_category_name", "product_category_name_english"
    },
}

FILE_NAMES = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

DATE_COLUMNS = {
    "orders": [
        "order_purchase_timestamp", "order_approved_at",
        "order_delivered_carrier_date", "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": ["shipping_limit_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"],
}

## 3. Verify file availability

In [ ]:
missing_files = [
    file_name
    for file_name in FILE_NAMES.values()
    if not (RAW_DATA_DIR / file_name).exists()
]

if missing_files:
    missing_text = "\n".join(f"- {name}" for name in missing_files)
    raise FileNotFoundError(
        "The following required files are missing from data/raw/:\n"
        f"{missing_text}"
    )

print("All required raw files were found.")

## 4. Load the datasets

Date fields are parsed during ingestion. The raw CSV files are not modified.

In [ ]:
def load_dataset(
    file_path: Path,
    date_columns: Iterable[str] | None = None,
) -> pd.DataFrame:
    """Load one CSV file and parse requested date columns."""
    dataframe = pd.read_csv(file_path, low_memory=False)

    if date_columns:
        for column in date_columns:
            if column in dataframe.columns:
                dataframe[column] = pd.to_datetime(
                    dataframe[column],
                    errors="coerce",
                )

    return dataframe


datasets: Dict[str, pd.DataFrame] = {}

for dataset_name, file_name in FILE_NAMES.items():
    datasets[dataset_name] = load_dataset(
        file_path=RAW_DATA_DIR / file_name,
        date_columns=DATE_COLUMNS.get(dataset_name),
    )

print("Loaded datasets:")
for dataset_name, dataframe in datasets.items():
    print(
        f"- {dataset_name:<22} "
        f"{dataframe.shape[0]:>8,} rows × {dataframe.shape[1]:>2} columns"
    )

## 5. Validate expected columns

In [ ]:
schema_validation_records = []

for dataset_name, expected_columns in EXPECTED_SCHEMAS.items():
    actual_columns = set(datasets[dataset_name].columns)
    missing_columns = sorted(expected_columns - actual_columns)
    additional_columns = sorted(actual_columns - expected_columns)

    schema_validation_records.append(
        {
            "dataset": dataset_name,
            "schema_valid": len(missing_columns) == 0,
            "missing_required_columns": ", ".join(missing_columns),
            "additional_columns": ", ".join(additional_columns),
        }
    )

schema_validation = pd.DataFrame(schema_validation_records)
schema_validation

In [ ]:
invalid_schemas = schema_validation.loc[~schema_validation["schema_valid"]]

if not invalid_schemas.empty:
    raise ValueError(
        "At least one dataset is missing required columns. "
        "Review the schema_validation table above."
    )

print("All dataset schemas passed the minimum-column validation.")

## 6. Create a dataset inventory

The inventory summarizes row counts, column counts, missing cells, exact duplicate rows, memory usage, and source-file size.

In [ ]:
inventory_records = []

for dataset_name, dataframe in datasets.items():
    file_path = RAW_DATA_DIR / FILE_NAMES[dataset_name]

    inventory_records.append(
        {
            "dataset": dataset_name,
            "source_file": file_path.name,
            "rows": len(dataframe),
            "columns": dataframe.shape[1],
            "missing_cells": int(dataframe.isna().sum().sum()),
            "missing_cell_percentage": round(
                100 * dataframe.isna().sum().sum() / max(dataframe.size, 1),
                4,
            ),
            "exact_duplicate_rows": int(dataframe.duplicated().sum()),
            "memory_mb": round(
                dataframe.memory_usage(deep=True).sum() / 1_048_576,
                3,
            ),
            "source_file_mb": round(
                file_path.stat().st_size / 1_048_576,
                3,
            ),
        }
    )

data_inventory = (
    pd.DataFrame(inventory_records)
    .sort_values("dataset")
    .reset_index(drop=True)
)

data_inventory

## 7. Validate key identifiers

Orders may appear more than once in the item and payment tables, so composite keys are used there.

In [ ]:
key_checks = pd.DataFrame(
    [
        {
            "dataset": "customers",
            "key": "customer_id",
            "duplicate_key_rows": int(
                datasets["customers"].duplicated(
                    subset=["customer_id"], keep=False
                ).sum()
            ),
        },
        {
            "dataset": "orders",
            "key": "order_id",
            "duplicate_key_rows": int(
                datasets["orders"].duplicated(
                    subset=["order_id"], keep=False
                ).sum()
            ),
        },
        {
            "dataset": "order_items",
            "key": "order_id + order_item_id",
            "duplicate_key_rows": int(
                datasets["order_items"].duplicated(
                    subset=["order_id", "order_item_id"], keep=False
                ).sum()
            ),
        },
        {
            "dataset": "payments",
            "key": "order_id + payment_sequential",
            "duplicate_key_rows": int(
                datasets["payments"].duplicated(
                    subset=["order_id", "payment_sequential"], keep=False
                ).sum()
            ),
        },
        {
            "dataset": "products",
            "key": "product_id",
            "duplicate_key_rows": int(
                datasets["products"].duplicated(
                    subset=["product_id"], keep=False
                ).sum()
            ),
        },
        {
            "dataset": "sellers",
            "key": "seller_id",
            "duplicate_key_rows": int(
                datasets["sellers"].duplicated(
                    subset=["seller_id"], keep=False
                ).sum()
            ),
        },
    ]
)

key_checks

## 8. Validate basic parent-child relationships

In [ ]:
def count_orphan_keys(
    child: pd.DataFrame,
    child_key: str,
    parent: pd.DataFrame,
    parent_key: str,
) -> int:
    """Count distinct non-null child keys missing from the parent table."""
    child_values = set(child[child_key].dropna().unique())
    parent_values = set(parent[parent_key].dropna().unique())
    return len(child_values - parent_values)


relationship_checks = pd.DataFrame(
    [
        {
            "relationship": "orders.customer_id -> customers.customer_id",
            "orphan_keys": count_orphan_keys(
                datasets["orders"], "customer_id",
                datasets["customers"], "customer_id"
            ),
        },
        {
            "relationship": "order_items.order_id -> orders.order_id",
            "orphan_keys": count_orphan_keys(
                datasets["order_items"], "order_id",
                datasets["orders"], "order_id"
            ),
        },
        {
            "relationship": "payments.order_id -> orders.order_id",
            "orphan_keys": count_orphan_keys(
                datasets["payments"], "order_id",
                datasets["orders"], "order_id"
            ),
        },
        {
            "relationship": "reviews.order_id -> orders.order_id",
            "orphan_keys": count_orphan_keys(
                datasets["reviews"], "order_id",
                datasets["orders"], "order_id"
            ),
        },
        {
            "relationship": "order_items.product_id -> products.product_id",
            "orphan_keys": count_orphan_keys(
                datasets["order_items"], "product_id",
                datasets["products"], "product_id"
            ),
        },
        {
            "relationship": "order_items.seller_id -> sellers.seller_id",
            "orphan_keys": count_orphan_keys(
                datasets["order_items"], "seller_id",
                datasets["sellers"], "seller_id"
            ),
        },
    ]
)

relationship_checks

## 9. Inspect representative samples

In [ ]:
for dataset_name in [
    "customers", "orders", "order_items", "payments", "reviews"
]:
    print(f"\n{dataset_name.upper()}")
    display(datasets[dataset_name].head(3))

## 10. Export validation artifacts

These files support reproducibility and will be used by the data-quality notebook.

In [ ]:
output_files = {
    "schema_validation.csv": schema_validation,
    "data_inventory.csv": data_inventory,
    "key_validation.csv": key_checks,
    "relationship_validation.csv": relationship_checks,
}

for file_name, dataframe in output_files.items():
    output_path = PROCESSED_DATA_DIR / file_name
    dataframe.to_csv(output_path, index=False)
    print(f"Created: {output_path.relative_to(PROJECT_ROOT)}")

## Ingestion summary

This notebook:

- loaded the raw relational datasets;
- preserved the original source files;
- parsed date and timestamp columns;
- validated minimum schemas;
- checked important identifiers;
- checked parent-child relationships;
- created reproducible validation artifacts.

### Next notebook

`02_data_quality_and_eda.ipynb` will:

1. analyze missing values and duplicates;
2. validate dates, prices, freight values, scores, and order statuses;
3. identify anomalous and inconsistent records;
4. create cleaned analytical tables;
5. perform the first exploratory visualizations.